In [119]:
import torch
from torch import nn
import torchvision
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import math
from torchmetrics.classification import MulticlassAccuracy
from timeit import default_timer as timer
from tqdm.notebook import tqdm
import sys

In [120]:
RANDOM_SEED = 42
BATCH_SIZE = 32

In [121]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"

In [122]:
train_data = datasets.FashionMNIST(
    root= "data",
    train=True,
    transform=ToTensor(),
    target_transform=None,
    download=True,
)

test_data = datasets.FashionMNIST(
    root= "data",
    train=False,
    transform=ToTensor(),
    target_transform=None,
    download=True,
)

In [123]:
train_data_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_data_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [124]:
class CVModel_2(nn.Module):
    def __init__(self, input_features, output_features, hidden_neurons):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_features, out_features=hidden_neurons),
            nn.Tanh(),
            nn.Linear(in_features=hidden_neurons, out_features=output_features),
            # nn.ReLU()
        )

    def forward(self, x):
        return self.layer_stack(x)

In [125]:
class_names = test_data.classes
class_names

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [126]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
cv_model = CVModel_2(input_features=28*28, hidden_neurons=10, output_features=len(class_names)).to(device)

In [127]:
optimizer = torch.optim.AdamW(cv_model.parameters(), lr=.001)
loss_fn = nn.CrossEntropyLoss()
acc_fn = MulticlassAccuracy(num_classes=len(class_names)).to(device)

In [128]:
def train_step(model: torch.nn.Module,
                optimizer: torch.optim.AdamW, 
                loss_fn: torch.nn.CrossEntropyLoss, 
                train_data_loader: torch.utils.data.DataLoader, 
                acc_fn: MulticlassAccuracy, 
                epoch, 
                device: torch.device = device):

    model.train()

    train_loss, train_acc = 0, 0

    pbar = tqdm(train_data_loader, desc=f"training step for the {epoch} epoch", leave=False)

    for X, y in pbar:

        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        train_logits = model(X)
        train_pred = torch.softmax(train_logits, dim=1).argmax(dim=1)

        train_loss_ = loss_fn(train_logits, y)
        train_loss_.backward()

        optimizer.step()

        train_loss_ = loss_fn(train_logits, y)
        train_loss += train_loss_.item()

        train_acc_ = acc_fn(train_pred, y)
        train_acc += train_acc_.item()

    train_acc /= len(train_data_loader)
    train_loss /= len(train_data_loader)

    pbar.set_postfix({"loss": f"{train_loss_.item():.4f}"})

    return train_loss, train_acc

In [129]:
def test_step(model, 
              loss_fn: torch.nn.CrossEntropyLoss, 
              test_data_loader: torch.utils.data.DataLoader, 
              acc_fn: MulticlassAccuracy, 
              epoch, 
              device: torch.device=device):

    model.eval()
    test_acc, test_loss = 0, 0

    with torch.inference_mode():

        pbar = tqdm(test_data_loader, desc=f"testing step for the {epoch} epoch", leave=False)

        for X, y in pbar:

            X, y = X.to(device), y.to(device)

            test_logits = model(X) 
            test_preds = torch.softmax(test_logits, dim=1).argmax(dim=1)

            test_loss_ = loss_fn(test_logits, y)
            test_loss += test_loss_.item()

            test_acc_ = acc_fn(test_preds, y)
            test_acc += test_acc_.item()

        test_acc /= len(test_data_loader)
        test_loss /= len(test_data_loader)

        pbar.set_postfix({"loss": f"{test_loss_.item():.4f}"})

    return test_loss, test_acc

In [130]:
def train_time(start: float,
               end: float,
               device: torch.device = None):
    total_time = end - start
    print(f"total training time {total_time:.3f} on the {device}")
    return total_time

In [132]:
torch.cuda.manual_seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

start_time = timer()
epochs = 3

epoch_pbar = tqdm(range(epochs), desc="Overall Progress")

for epoch in epoch_pbar:
    train_loss, train_acc = train_step(cv_model, optimizer, loss_fn, train_data_loader, acc_fn, epoch, device)
    test_loss, test_acc = test_step(cv_model, loss_fn, test_data_loader, acc_fn, epoch, device)

    epoch_pbar.set_postfix({
            "train_loss": f"{train_loss:.4f}",
            "test_acc": f"{test_acc * 100:.2f}%"
        })

end_time = timer()

train_time(start=start_time, end=end_time)

Overall Progress:   0%|          | 0/3 [00:00<?, ?it/s]

training step for the 0 epoch:   0%|          | 0/1875 [00:00<?, ?it/s]

testing step for the 0 epoch:   0%|          | 0/313 [00:00<?, ?it/s]

training step for the 1 epoch:   0%|          | 0/1875 [00:00<?, ?it/s]

testing step for the 1 epoch:   0%|          | 0/313 [00:00<?, ?it/s]

training step for the 2 epoch:   0%|          | 0/1875 [00:00<?, ?it/s]

testing step for the 2 epoch:   0%|          | 0/313 [00:00<?, ?it/s]

total training time 42.088 on the None


42.08792979999998

In [133]:
print(f"the train acc :{train_acc*100}% with a loss of {train_loss}")
print(f"the test acc :{test_acc*100}% with a loss of {test_loss}")

the train acc :84.75937837600708% with a loss of 0.39825885720650356
the test acc :83.29852269099544% with a loss of 0.4456851683057154


In [138]:
def eval_model(model: torch.nn.Module, 
              loss_fn: torch.nn.CrossEntropyLoss, 
              test_data_loader: torch.utils.data.DataLoader, 
              acc_fn: MulticlassAccuracy, 
              device: torch.device=device):

    model.eval()
    test_acc, test_loss = 0, 0

    with torch.inference_mode():

        pbar = tqdm(test_data_loader, desc=f"model evaluation", leave=False)

        for X, y in pbar:

            X, y = X.to(device), y.to(device)

            test_logits = model(X) 
            test_preds = torch.softmax(test_logits, dim=1).argmax(dim=1)

            test_loss_ = loss_fn(test_logits, y)
            test_loss += test_loss_.item()

            test_acc_ = acc_fn(test_preds, y)
            test_acc += test_acc_.item()

        test_acc /= len(test_data_loader)
        test_loss /= len(test_data_loader)

        pbar.set_postfix({"loss": f"{test_loss_.item():.4f}"})

    return {
            "model_name": model.__class__.__name__,
            "model_loss": test_loss,
            "model_acc": test_acc
        }

In [139]:
eval_model(cv_model, loss_fn, test_data_loader, acc_fn, device)

model evaluation:   0%|          | 0/313 [00:00<?, ?it/s]

{'model_name': 'CVModel_2',
 'model_loss': 0.4456851683057154,
 'model_acc': 0.8329852269099544}